# StanceEval2026 Kaggle Runner

This notebook is only a thin execution interface. The training, evaluation, and submission logic lives in `src/` and `scripts/`.

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import subprocess
import sys
from pathlib import Path

search_roots = [Path.cwd()]
if Path('/kaggle/input').exists():
    search_roots.append(Path('/kaggle/input'))

requirements = None
for root in search_roots:
    for path in root.rglob('requirements.txt'):
        requirements = path
        break
    if requirements is not None:
        break

if requirements is None:
    raise FileNotFoundError('requirements.txt was not found.')

print('Installing from:', requirements)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])

Installing from: c:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\requirements.txt


0

In [3]:
import os
import shutil
from pathlib import Path

is_kaggle = Path('/kaggle/working').exists()

def find_project_dir():
    candidates = [Path.cwd()]
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob('configs/config.yaml'))
    for candidate in candidates:
        if (candidate / 'configs/config.yaml').exists() and (candidate / 'scripts').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find project files. Expected configs/config.yaml and scripts/.')

project_dir = find_project_dir()
if is_kaggle and str(project_dir).startswith('/kaggle/input'):
    work_project = Path('/kaggle/working/stanceeval_project')
    if work_project.exists():
        shutil.rmtree(work_project)
    shutil.copytree(project_dir, work_project)
    project_dir = work_project

os.chdir(project_dir)

os.environ.setdefault('STANCEEVAL_DATA_DIR', str(project_dir))
os.environ.setdefault('STANCEEVAL_OUTPUT_DIR', '/kaggle/working/outputs' if is_kaggle else 'outputs')
os.environ.setdefault('STANCEEVAL_RESULTS_DIR', '/kaggle/working/results' if is_kaggle else 'results')
os.environ.setdefault('STANCEEVAL_BEST_MODEL_DIR', '/kaggle/working/outputs/best_model' if is_kaggle else 'outputs/best_model')

print('Data:', os.environ['STANCEEVAL_DATA_DIR'])
print('Outputs:', os.environ['STANCEEVAL_OUTPUT_DIR'])
print('Results:', os.environ['STANCEEVAL_RESULTS_DIR'])
print('Project:', project_dir)

Data: C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main
Outputs: outputs
Results: results
Project: C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main


In [4]:
!python scripts/smoke_test.py --config configs/config.yaml --base-model-id aubmindlab/bert-base-arabertv02-twitter --verbose

[1/12] Loading configuration...
[1/12] Loading configuration: SUCCESS (0.01s)
[2/12] Validating files and folders...
[2/12] Validating files and folders: FAIL (0.00s)
Stage failed: Validating files and folders
Error: FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv']

Pipeline Status:
✘ Config - FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv']


Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\pipeline_logging.py", line 46, in stage
    yield started
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\smoke_test.py", line 62, in main
    require_files([train_path])
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\validation.py", line 23, in require_files
    raise FileNotFoundError(f"Missing required file(s): {missing}")
FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv']
Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\smoke_test.py", line 144, in <module>
    main()
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\smoke_test.py", line 62, in main
    require_files([train_path])
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\

In [5]:
!python scripts/compare_base_models.py --config configs/config.yaml --verbose

[1/6] Loading configuration...
[1/6] Loading configuration: SUCCESS (0.02s)
[2/6] Validating paths...
[2/6] Validating paths: FAIL (0.00s)
Stage failed: Validating paths
Error: FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv', 'C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\dev.csv']

Pipeline Status:
✔ Config
✘ Dataset - FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv', 'C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\dev.csv']


Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\pipeline_logging.py", line 46, in stage
    yield started
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\compare_base_models.py", line 42, in main
    require_files([train_path, dev_path])
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\validation.py", line 23, in require_files
    raise FileNotFoundError(f"Missing required file(s): {missing}")
FileNotFoundError: Missing required file(s): ['C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\train.csv', 'C:\\Users\\njood\\Downloads\\stanceeval\\stanceeval.github.io-main\\dev.csv']
Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\compare_base_models.py", line 116, in <module>
    main()
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\compare_base_models.py", line 42

In [6]:
!python scripts/generate_submissions.py --config configs/config.yaml --verbose

[1/10] Loading configuration...
[1/10] Loading configuration: SUCCESS (0.02s)
[2/10] GPU diagnostics...
CUDA available: True
PyTorch version: 2.10.0+cu126
Transformers version: 5.0.0
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory total MB: 8187
GPU memory allocated MB: 0
GPU memory reserved MB: 0
[2/10] GPU diagnostics: SUCCESS (0.03s)
[3/10] Resolving input and output paths...
[3/10] Resolving input and output paths: FAIL (0.00s)
Stage failed: Resolving input and output paths
Error: FileNotFoundError: Could not find seen test file. Tried C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\test_seen.csv

Pipeline Status:
✘ Config - FileNotFoundError: Could not find seen test file. Tried C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\test_seen.csv
✔ GPU


Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\src\pipeline_logging.py", line 46, in stage
    yield started
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\generate_submissions.py", line 67, in main
    seen_test_path = resolve_test_path(
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\generate_submissions.py", line 29, in resolve_test_path
    raise FileNotFoundError(
FileNotFoundError: Could not find seen test file. Tried C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\test_seen.csv
Traceback (most recent call last):
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\generate_submissions.py", line 144, in <module>
    main()
  File "C:\Users\njood\Downloads\stanceeval\stanceeval.github.io-main\scripts\generate_submissions.py", line 67, in main
    seen_test_path = resolve_test_path(
  File "C:\Users\njood\Downloads\stanceeval\

In [7]:
import pandas as pd
from pathlib import Path

results_dir = Path(os.environ['STANCEEVAL_RESULTS_DIR'])
for name in ['base_model_comparison.csv']:
    path = results_dir / name
    if path.exists():
        display(pd.read_csv(path))

print('submission_seen:', results_dir / 'submission_seen.csv')
print('submission_unseen:', results_dir / 'submission_unseen.csv')
print('submission_zip:', results_dir / 'submission_files.zip')

submission_seen: results\submission_seen.csv
submission_unseen: results\submission_unseen.csv
submission_zip: results\submission_files.zip
